# Batch Feature Extractor (Simulation vs CAD)
This notebook isolates specific feature point clouds, calculates their exact Chamfer Distance noise, and saves them to disk to serve as the training dataset for PointNet.

In [4]:
import os
import glob
import pandas as pd
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. CONFIGURATION
# ==========================================
# WORKPIECES = ["TH0011AV", "TH0012AV", "TH0021AV", "TH0022AV", "TH0031AV", "TH0032AV"]
WORKPIECES = ["TH0011AV", "TH0012AV", "TH0021AV", "TH0022AV", "TH0031AV", "TH0032AV", "TH0041AV", "TH0042AV", "TH0051AV", "TH0052AV", "TH0061AV", "TH0062AV", "TH0071AV", "TH0072AV"]
# WORKPIECES = ["TH0011AV", "TH0012AV"]
EXPERIMENT = "test_8_simulation2"
DATASET_NAME = EXPERIMENT
DISTANCE_TRESHOLD = 2.0
NUMBER_OF_POINTS = 5000

results = []

for workpiece in WORKPIECES:
    print(f"\n==========================================")
    print(f"ANALYZING & EXTRACTING SURFACES: {workpiece}")
    print(f"==========================================")
    
    PROCESSED_DIR = f"processed_data/{EXPERIMENT}/{workpiece}"
    SIM_DIR = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{workpiece}"
    WORKPIECE_DIR = f"workpiece/{workpiece}"
    
    # Setup the new Export Directory for PointNet
    EXPORT_DIR = PROCESSED_DIR
    os.makedirs(EXPORT_DIR, exist_ok=True)
    
    if not os.path.exists(PROCESSED_DIR):
        print(f"Skipping {workpiece} - No processed data found.")
        continue
        
    # Find all surface files dynamically
    surface_files = glob.glob(os.path.join(WORKPIECE_DIR, "surface*.stl"))
    if not surface_files:
        print(f"No surface features found for {workpiece}.")
        continue
        
    # Find all simulated point clouds
    # Find all NOISY simulated point clouds (to calculate error)
    SIMULATION_OUTPUT_DIR = f"simulation/{EXPERIMENT}/{workpiece}"
    pcd_files = glob.glob(os.path.join(SIMULATION_OUTPUT_DIR, "viewpoint_simulated_noise_*.pcd"))
    if not pcd_files:
        # Fallback if Phase 3 was actually run and files were copied
        pcd_files = glob.glob(os.path.join(PROCESSED_DIR, "viewpoint_simulated_noise_*.pcd"))
    # Filter out leftover crops from previous runs
    pcd_files = [f for f in pcd_files if "_surface" not in f]
    if not pcd_files:
        print(f"No simulated point clouds found for {workpiece}.")
        continue
        
    for surface_path in surface_files:
        surface_name = os.path.basename(surface_path)
        surface_clean_name = surface_name.replace('.stl', '')
        print(f"--> Evaluating {surface_name}...")
        
        # Load feature mesh -> Point Cloud
        feature_mesh = o3d.io.read_triangle_mesh(surface_path)
        feature_mesh.compute_vertex_normals()
        feature_pcd = feature_mesh.sample_points_poisson_disk(number_of_points=NUMBER_OF_POINTS)
        
        count_hits = 0
        
        for noisy_pcd_path in pcd_files:
            viewpoint_name_noisy = os.path.basename(noisy_pcd_path).replace('.pcd', '')
            view_idx = viewpoint_name_noisy.replace('viewpoint_simulated_noise_', '')
            
            perfect_pcd_path = os.path.join(SIM_DIR, f"viewpoint_simulated_{view_idx}.pcd")
            if not os.path.exists(perfect_pcd_path):
                print(f"Missing perfect CAD for view {view_idx}, skipping.")
                continue
            
            viewpoint_name_perfect = f"viewpoint_simulated_{view_idx}"
            
            noisy_pcd = o3d.io.read_point_cloud(noisy_pcd_path)
            perfect_pcd = o3d.io.read_point_cloud(perfect_pcd_path)
            
            # 1. Evaluate NOISY point cloud to calculate Chamfer Distance
            dists_noisy = np.asarray(noisy_pcd.compute_point_cloud_distance(feature_pcd))
            intersection_indices_noisy = np.where(dists_noisy < DISTANCE_TRESHOLD)[0]
            if len(intersection_indices_noisy) < 10:
                continue
                
            intersection_noisy_pcd = noisy_pcd.select_by_index(intersection_indices_noisy)
            
            dists_s2c = np.asarray(intersection_noisy_pcd.compute_point_cloud_distance(feature_pcd))
            dists_c2s_full = np.asarray(feature_pcd.compute_point_cloud_distance(intersection_noisy_pcd))
            valid_c2s_indices = np.where(dists_c2s_full < DISTANCE_TRESHOLD)[0]
            if len(valid_c2s_indices) == 0:
                continue
            dists_c2s = dists_c2s_full[valid_c2s_indices]
            chamfer_dist = np.mean(dists_s2c) + np.mean(dists_c2s)
            
            # 2. Evaluate PERFECT point cloud to serve as the ML Input
            dists_perfect = np.asarray(perfect_pcd.compute_point_cloud_distance(feature_pcd))
            intersection_indices_perfect = np.where(dists_perfect < DISTANCE_TRESHOLD)[0]
            if len(intersection_indices_perfect) < 10:
                continue
                
            intersection_perfect_pcd = perfect_pcd.select_by_index(intersection_indices_perfect)
            count_hits += 1
            
            # 3. Export the PERFECT Feature Crop to Disk for PointNet!
            export_filename = f"{viewpoint_name_perfect}_{surface_clean_name}.pcd"
            export_path = os.path.join(EXPORT_DIR, export_filename)
            o3d.io.write_point_cloud(export_path, intersection_perfect_pcd)
            
            # 4. Save EXACT perfect format string to CSV mapped to the noisy chamfer distance
            relative_path = f"{workpiece}/{export_filename}"
            results.append({
                "filename": relative_path,
                "dist_s2r": np.mean(dists_s2c),  # Legacy placeholder
                "dist_r2s": np.mean(dists_c2s),  # Legacy placeholder
                "chamfer_value": chamfer_dist,
                "asymmetry_value": 0.0           # Legacy placeholder
            })
            
        print(f"    (Exported {count_hits} cropped point clouds for {surface_name})")

print("\n==========================================")
print("DONE EXTRACTING FEATURES!")
print("==========================================")

# ==========================================
# 2. EXPORT POINTNET METADATA CSV
# ==========================================
df_results = pd.DataFrame(results)

CSV_EXPORT_DIR = f"processed_data/{DATASET_NAME}"
os.makedirs(CSV_EXPORT_DIR, exist_ok=True)
csv_path = os.path.join(CSV_EXPORT_DIR, "metadata.csv")
df_results.to_csv(csv_path, index=False)

print(f"Saved {len(df_results)} point cloud labels to {csv_path}")



ANALYZING & EXTRACTING SURFACES: TH0011AV
--> Evaluating surface0.stl...
    (Exported 432 cropped point clouds for surface0.stl)
--> Evaluating surface1.stl...
    (Exported 432 cropped point clouds for surface1.stl)

ANALYZING & EXTRACTING SURFACES: TH0012AV
--> Evaluating surface0.stl...
    (Exported 432 cropped point clouds for surface0.stl)
--> Evaluating surface1.stl...
    (Exported 432 cropped point clouds for surface1.stl)

ANALYZING & EXTRACTING SURFACES: TH0021AV
--> Evaluating surface0.stl...
    (Exported 432 cropped point clouds for surface0.stl)
--> Evaluating surface1.stl...
    (Exported 432 cropped point clouds for surface1.stl)

ANALYZING & EXTRACTING SURFACES: TH0022AV
--> Evaluating surface0.stl...
    (Exported 216 cropped point clouds for surface0.stl)
--> Evaluating surface1.stl...
    (Exported 216 cropped point clouds for surface1.stl)

ANALYZING & EXTRACTING SURFACES: TH0031AV
--> Evaluating surface0.stl...
    (Exported 216 cropped point clouds for surface

In [10]:
# ==========================================
# 3. INTERACTIVE VISUALIZATION
# ==========================================
import math
import os
import copy
import numpy as np
import open3d as o3d

WORKPIECE = "TH0011AV"
SURFACE_FILE = "surface0.stl"
VIEWPOINT_IDX = 128
EXPERIMENT = "test_8_simulation2"
DISTANCE_TRESHOLD = 2.0

print(f"Loading {WORKPIECE} - {SURFACE_FILE} - Viewpoint {VIEWPOINT_IDX} for interactive validation...")

PROCESSED_DIR = f"processed_data/{EXPERIMENT}/{WORKPIECE}"
WORKPIECE_DIR = f"workpiece/{WORKPIECE}"

pcd_path = os.path.join(PROCESSED_DIR, f"viewpoint_simulated_noise_{VIEWPOINT_IDX}.pcd")
surface_path = os.path.join(WORKPIECE_DIR, SURFACE_FILE)

def default_visualization(geometries, window_name="Default Visualization", zoom=1.0):
    azimuth_deg = -45
    elevation_deg = -135
    az = math.radians(azimuth_deg)
    el = math.radians(elevation_deg)
    front = np.array([math.cos(el) * math.cos(az), math.cos(el) * math.sin(az), math.sin(el)])
    front = -front
    if isinstance(geometries, list) and len(geometries) > 0:
        lookat = geometries[0].get_center()
    else:
        lookat = [0, 0, 0]
    up = [0, 0, 1]
    o3d.visualization.draw_geometries(geometries, window_name=window_name, width=1024, height=768, lookat=lookat, up=up, front=front, zoom=zoom)

try:
    # Load feature mesh
    feature_mesh = o3d.io.read_triangle_mesh(surface_path)
    feature_mesh.compute_vertex_normals()
    feature_pcd = feature_mesh.sample_points_poisson_disk(number_of_points=5000)
    feature_pcd.paint_uniform_color([1, 0, 0]) # Red

    # Load NOISY simulated PCD
    noisy_pcd = o3d.io.read_point_cloud(pcd_path)
    noisy_pcd.paint_uniform_color([0.5, 0.5, 0.5]) # Gray

    # Calculate NOISY Intersection
    dists_noisy = np.asarray(noisy_pcd.compute_point_cloud_distance(feature_pcd))
    intersection_indices_noisy = np.where(dists_noisy < DISTANCE_TRESHOLD)[0]
    intersection_noisy_pcd = noisy_pcd.select_by_index(intersection_indices_noisy)
    intersection_noisy_pcd.paint_uniform_color([0, 0, 1]) # Blue
    
    # Load PERFECT simulated PCD
    perfect_pcd_path = os.path.join(PROCESSED_DIR.replace('processed_data', 'viewpoints_candidate/testing_data'), f"viewpoint_simulated_{VIEWPOINT_IDX}.pcd")
    perfect_pcd = o3d.io.read_point_cloud(perfect_pcd_path)
    perfect_pcd.paint_uniform_color([0.5, 0.5, 0.5]) # Gray
    
    # Calculate PERFECT Intersection
    dists_perf = np.asarray(perfect_pcd.compute_point_cloud_distance(feature_pcd))
    intersection_indices_perf = np.where(dists_perf < DISTANCE_TRESHOLD)[0]
    intersection_perf_pcd = perfect_pcd.select_by_index(intersection_indices_perf)
    intersection_perf_pcd.paint_uniform_color([0, 1, 0]) # Green

    print("\nRed: Perfect CAD Feature")
    print("Gray: Full Noisy Simulated Viewpoint")
    print("Blue: Noisy Simulated Intersect")
    print("Green: Perfect Simulated Intersect")
    
    default_visualization([noisy_pcd, feature_pcd], window_name="Full Scene (Gray=Noisy Sim, Red=CAD Feature)")
    default_visualization([intersection_perf_pcd, intersection_noisy_pcd], window_name="Intersections (Green=Perfect, Blue=Noisy)")

except Exception as e:
    print(f"Error loading or visualizing point clouds: {e}")


Loading TH0011AV - surface0.stl - Viewpoint 128 for interactive validation...

Red: Perfect CAD Feature
Gray: Full Noisy Simulated Viewpoint
Blue: Noisy Simulated Intersect
Green: Perfect Simulated Intersect
